In [4]:
from glob import glob as glob
import os
import json
import mne
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import h5py
from pathlib import Path
import logging
import hydra
from omegaconf import DictConfig, OmegaConf
from scipy import signal, stats
import csv
import re
from tqdm import tqdm as tqdm

In [137]:
# raw = mne.io.read_raw_edf(data_fpath)

# read from h5

In [22]:
out_path = "/storage/czw/mgh_focal_h5s_scaled/1/9a2b63fa-5ca7-4272-a6b6-d7d3fc647d3a_scaled.h5"
i = 1

with h5py.File(out_path, 'r') as f:
    offsets = f['data']['offsets'][:]
    cal = f['data']['cal'][:]
    gains = f['data']['gains'][:]
    scaled = f['data'][f'channel_{i}'][:]


In [21]:
scaled = scaled.astype(np.float64)
scaled

array([[-47., -19., -14., ...,   0.,   0.,   0.]])

In [11]:
scaled = scaled.astype(np.float64)
scaled *= cal[i]
scaled += offsets[i]
scaled *= gains[i]

In [12]:
scaled.dtype

dtype('float64')

In [13]:
scaled

array([[ 6.89861753e-05,  1.13913588e-04,  1.55916732e-04, ...,
        -1.32921340e-07, -1.32921340e-07, -1.32921340e-07]])

# read from edf

In [14]:
raw = mne.io.read_raw_edf(data_fpath)

NameError: name 'data_fpath' is not defined

In [145]:
edf_data, times = raw[[i], :]

In [151]:
edf_data

array([[ 6.89861753e-05,  1.13913588e-04,  1.55916732e-04, ...,
        -1.32921340e-07, -1.32921340e-07, -1.32921340e-07]],
      shape=(1, 9588160))

In [156]:
np.isclose(scaled, edf_data).all()

np.True_

# scratch work

In [4]:
raw.info

<Info | 8 non-empty values
 bads: []
 ch_names: RAMY1, RAMY2, RAMY3, RAMY4, RAMY5, RAMY6, RAMY7, RAMY8, RAMY9, ...
 chs: 276 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 512.0 Hz
 meas_date: 2001-01-01 00:00:00 UTC
 nchan: 276
 projs: []
 sfreq: 1024.0 Hz
 subject_info: <subject_info | his_id: XXX-XX-XX, sex: 0, last_name: cb37fb19-da50-4a59-8220-b68b18978ae5, birthday: 2001-01-01>
>

In [5]:
raw.info["physical_max"]

KeyError: 'physical_max'

In [6]:
raw.info

<Info | 8 non-empty values
 bads: []
 ch_names: RAMY1, RAMY2, RAMY3, RAMY4, RAMY5, RAMY6, RAMY7, RAMY8, RAMY9, ...
 chs: 276 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 512.0 Hz
 meas_date: 2001-01-01 00:00:00 UTC
 nchan: 276
 projs: []
 sfreq: 1024.0 Hz
 subject_info: <subject_info | his_id: XXX-XX-XX, sex: 0, last_name: cb37fb19-da50-4a59-8220-b68b18978ae5, birthday: 2001-01-01>
>

In [10]:
raw = mne.io._read_edf_header(data_fpath)

AttributeError: No mne.io attribute _read_edf_header

In [12]:
raw.info["cals"]

KeyError: 'cals'

In [14]:
edf_data, times = raw[[0], :]

In [29]:
physical_min, physical_max = edf_data.min(), edf_data.max()
digital_min, digital_max = np.iinfo(np.int16).min, np.iinfo(np.int16).max

In [30]:
scaled = (edf_data - physical_min)*((digital_max-digital_min)/(physical_max-physical_min)) + digital_min

In [40]:
unscaled = (scaled - digital_min)*((physical_max-physical_min)/(digital_max-digital_min)) + physical_min

In [39]:
np.abs(unscaled - edf_data).mean()

np.float64(6.843766628924158e-19)

In [34]:
edf_data.dtype

dtype('float64')

In [19]:
import sys

In [20]:
int_max = sys.maxsize

In [21]:
int_max

9223372036854775807

In [23]:
max_int16 = np.iinfo(np.int16).max
print(f"Maximum of int32: {max_int16}")


Maximum of int32: 32767


In [25]:
min_int16 = np.iinfo(np.int16).min
print(f"Minimum of int32: {min_int16}")


Minimum of int32: -32768


In [41]:
raw.info.keys()

dict_keys(['acq_pars', 'acq_stim', 'ctf_head_t', 'description', 'dev_head_t', 'dev_ctf_t', 'dig', 'experimenter', 'utc_offset', 'device_info', 'file_id', 'highpass', 'hpi_subsystem', 'kit_system_id', 'helium_info', 'line_freq', 'lowpass', 'meas_date', 'meas_id', 'proj_id', 'proj_name', 'subject_info', 'xplotter_layout', 'gantry_angle', 'bads', 'chs', 'comps', 'events', 'hpi_meas', 'hpi_results', 'projs', 'proc_history', 'custom_ref_applied', 'sfreq', 'ch_names', 'nchan'])

In [42]:
raw._cals

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1.

In [62]:
offsets = raw._raw_extras[0]['offsets']
gains = raw._raw_extras[0]['units']
cal = raw._raw_extras[0]['cal']
assert len(offsets) == raw.info['nchan'] and len(gains) == raw.info['nchan'] and len(cal) == raw.info['nchan']

In [102]:
idx = 0
unscaled = edf_data.copy()
unscaled /= gains[idx]
unscaled -= offsets[idx]
unscaled /= cal[idx]
unscaled_full = unscaled.copy()
unscaled = np.round(unscaled).astype(np.int16)

In [103]:
unscaled

array([[-276, -463, -639, ...,    0,    0,    0]],
      shape=(1, 9588160), dtype=int16)

In [104]:
unscaled_full[~np.isclose(unscaled_full, unscaled)]

array([], dtype=float64)

In [105]:
unscaled[~np.isclose(unscaled_full, unscaled)]

array([], dtype=int16)

In [87]:
scaled = unscaled.copy().astype(np.float64)
scaled *= cal[idx]
scaled += offsets[idx]
scaled *= gains[idx]


In [88]:
np.abs(scaled - edf_data).sum()

np.float64(0.07170069492637107)

In [85]:
np.isclose(scaled, edf_data).all()

np.False_

In [79]:
offsets

array([-1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -1.32921340e-01, -1.32921340e-01, -1.32921340e-01,
       -1.32921340e-01, -